# Асинхронность в Python

![alt-текст](img/async_loop.png)

Объект Корутина - функции, которые могут быть приостановлены и возобновлены

In [ ]:
# Пример работы корутины в Python с использованием asyncio

import asyncio

# НЕ ПРАВИЛЬНО
async def example_coroutine():
    print("Hello from coroutine!")

example_coroutine()  # создан объект корутины example_coroutine(), корутина не запущена!

# ПРАВИЛЬНО
async def example_coroutine():
    print("Hello from coroutine!")

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await example_coroutine()  # корутина example_coroutine() запущена

Hello from coroutine!


/var/folders/ww/my_rn7cx4g3_6y5ybtlvj26r0000gn/T/ipykernel_76615/562275763.py:7: RuntimeWarning: coroutine 'example_coroutine' was never awaited
  example_coroutine()  # создан объект корутины example_coroutine(), корутина не запущена!


Объек Task - представляют собой обертки для корутин и используются для их конкурентного выполнения

In [ ]:
# Пример создания и запуска задачи

import asyncio

async def example_coroutine():
    await asyncio.sleep(1)
    print("Hello from coroutine!")

async def main():
    task = asyncio.create_task(example_coroutine())  # создаем задачу из корутины example_coroutine()
    await task # ждем выполнения задачи

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await main()

Hello from coroutine!


In [5]:
# Пример создания и запуска нескольких задач

import asyncio

async def example_coroutine():
    await asyncio.sleep(1)
    print("Hello from coroutine!")

async def main():
    tasks = []
    for _ in range(10):
        task = asyncio.create_task(example_coroutine())  # создаем 10 задач
        tasks.append(task)  # добавляем все задачи в список tasks
    await asyncio.gather(*tasks)  # запускаем все задачи из списка tasks

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await main()

Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!
Hello from coroutine!


Future - результат асинхронной операции, который появится в будущем

Объект Future/Task может находиться в одном из трёх состояний:
- Pending (ожидание) -  в этом состоянии операция, связанная с объектом Future , еще не выполнена
- Completed (завершён) - когда операция завершена, объект Future переходит в состояние "завершено"
- Cancelled (отменён) - если операция была отменена, объект Future переходит в состояние "отменено"

In [6]:
# Пример использования Future для получения результата из корутины

import asyncio


async def simulate_long_running_task(name, delay, future: asyncio.Future):
    print(f"Задача '{name}' началась, будет выполнена за {delay} секунд.")
    await asyncio.sleep(delay)
    result = f"Результат задачи '{name}'"
    print(f"Задача '{name}' завершена.")
    future.set_result(result)  # Устанавливаем результат для Future объекта


async def main():
    # Создаем объект Future
    future = asyncio.Future()

    # Запускаем корутину, передаем Future объект в функцию
    await simulate_long_running_task("Задача1", 3, future)

    # Получаем результат выполнения задачи
    result = future.result()
    print(f"Результат Future: {result}")

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await main()

Задача 'Задача1' началась, будет выполнена за 3 секунд.
Задача 'Задача1' завершена.
Результат Future: Результат задачи 'Задача1'


Правила и особенности использования `await`:
- `await` - это ключевое слово, которое используется для переключения между awaitable объектами
- `await` нельзя использовать с не awaitable объектами
- `await` нельзя использовать вне корутин
- `await` блокирует выполнение корутины
- `await` возвращает результат
- корутина должна содержать `await`, чтобы реализовать преимущества asyncio

Цикл событий - работает в одном потоке и занимается организацией задач: он отмечает, когда какая задача должна начаться, и запускает её в нужный момент.

![event_loop.png](img/event_loop.png)

Event loop обрабатывает события следующим образом:
- Сбор событий: Цикл событий отслеживает и собирает асинхронные события и задачи, которые нужно выполнить. Это включает в себя операции ввода/вывода, таймеры и запланированные асинхронные задачи. События и задачи могут поступать из различных источников и добавляются в очередь на обработку.

- Определение и распределение задач: Когда событие готово к обработке, event loop определяет соответствующую асинхронную задачу или корутину для его обработки. Приоритеты событий обычно определяются автоматически системой.

- Выполнение задач: Event loop запускает задачу и управляет её выполнением. Если задача асинхронна, она может быть приостановлена (await) для ожидания других событий (например, ответа от сервера) без блокировки цикла событий. Таким образом, event loop может переключаться между задачами, эффективно распределяя время процессора.

- Ожидание и завершение событий: После инициации задачи event loop продолжает отслеживать и обрабатывать другие входящие события. Он циклически возвращается к шагу сбора событий, пока есть активные или ожидающие задачи.

- Обработка ошибок: Если во время выполнения задачи возникает ошибка, она должна быть обработана внутри самой задачи или через механизмы обработки исключений asyncio. Цикл событий продолжит работу, если исключение будет корректно обработано. Необработанные исключения могут прервать выполнение задачи, но не обязательно останавливают сам цикл событий.

- Перезапуск цикла: Цикл событий продолжает работать до тех пор, пока не будет остановлен явно, либо до тех пор, пока не завершатся все задачи. Это позволяет приложению оставаться реактивным и обрабатывать события в реальном времени.